# Example 8.7.1 / 1: Count What a Declaration Generates

Companion notebook for **8.7.1 Efficiency and Large Models** in the AMPL lab Documentation, section "Count What a Declaration Generates". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Count What a Declaration Generates

A declaration over `I cross J cross K` creates up to `card(I)*card(J)*card(K)` instances before
filters and sparse membership are applied. In this version of the steel model, products are
made at two mills, but not every mill makes every product: the set `CAN_MAKE` lists the pairs
that exist. This model makes the common mistake: it indexes `Make` over every pair of `MILL` and
`PROD` and then forces the impossible pairs to zero with `Not_Made`. In `data.py` and `run.py`,
`ampl` is an `AMPL` object that has already read `model.mod`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                    # products
set MILL;                                    # mills
set CAN_MAKE within {MILL, PROD};            # pairs that can actually be produced
param rate {CAN_MAKE} > 0;                   # tons per hour at a mill
param avail {MILL} >= 0;                     # hours available at each mill
param profit {PROD};                         # profit per ton
var Make {MILL, PROD} >= 0;                  # dense: every pair, possible or not
maximize Total_Profit: sum {m in MILL, p in PROD} profit[p] * Make[m, p];
# hours used at each mill fit the hours available
subject to Time {m in MILL}:
    sum {(m, p) in CAN_MAKE} (1 / rate[m, p]) * Make[m, p] <= avail[m];
# the anti-pattern: impossible pairs forced to zero
subject to Not_Made {(m, p) in {MILL, PROD} diff CAN_MAKE}: Make[m, p] = 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.set["MILL"] = ["east", "west"]
ampl.set["CAN_MAKE"] = [("east", "bands"), ("east", "coils"), ("west", "plate")]
ampl.param["rate"] = {("east", "bands"): 200, ("east", "coils"): 140, ("west", "plate"): 160}
ampl.param["avail"] = {"east": 40, "west": 20}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}

In [5]:
# run.py
print("pairs:", ampl.get_value("card({MILL, PROD})"), "possible:", ampl.get_value("card(CAN_MAKE)"))
ampl.option["show_stats"] = 1
ampl.option["solver_msg"] = 0
ampl.solve(solver="highs")
print(ampl.get_value("_nvars"), "variables generated,", ampl.get_value("_snvars"), "sent to the solver")

pairs: 6 possible: 3

Presolve eliminates 4 constraints and 3 variables.
Adjusted problem:
3 variables, all linear
1 constraint, all linear; 2 nonzeros
	1 inequality constraint
1 linear objective; 3 nonzeros.

6 variables generated, 3 sent to the solver


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
set MILL := east west;
param: CAN_MAKE: rate :=
  east bands 200
  east coils 140
  west plate 160 ;
param avail := east 40  west 20;
param profit := bands 25  coils 30  plate 29;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display card({MILL, PROD}), card(CAN_MAKE);
option show_stats 1;
option solver_msg 0;
option solver highs;
solve;
display _nvars, _snvars;

card({MILL, PROD} ) = 6
card(CAN_MAKE) = 3


Presolve eliminates 4 constraints and 3 variables.
Adjusted problem:
3 variables, all linear
1 constraint, all linear; 2 nonzeros
	1 inequality constraint
1 linear objective; 3 nonzeros.

_nvars = 6
_snvars = 3

